# Hospital OPD Appointment and Billing Analytics

This notebook provides a complete SQL solution for the **Hospital OPD Appointment and Billing Analytics** assignment.
It covers schema creation, table design, sample data, reporting views, and business question queries.

## Approach

We will:
1. Create the database/schema `HOSPITAL_OPD_DB`.
2. Create the tables `PATIENTS`, `DEPARTMENTS`, `DOCTORS`, `APPOINTMENTS`, and `BILLING`.
3. Insert sample data.
4. Create the reporting view `V_OPD_APPOINTMENT_ANALYTICS`.
5. Create the summary view `V_DEPARTMENT_DAILY_REVENUE`.
6. Write SQL queries for all required business questions.

> Note: The SQL below uses standard relational database syntax and is suitable for learning/demo use. Minor datatype syntax may vary slightly across MySQL, PostgreSQL, SQL Server, or SAP HANA.

In [ ]:
# SQL script as a Python multiline string so the notebook remains executable and easy to export.
sql_script = r'''
-- =====================================================
-- 1. CREATE DATABASE / SCHEMA
-- =====================================================
CREATE SCHEMA IF NOT EXISTS HOSPITAL_OPD_DB;

-- Set schema context if supported by your database
-- For MySQL use: USE HOSPITAL_OPD_DB;
-- For PostgreSQL use: SET search_path TO HOSPITAL_OPD_DB;
-- For SAP HANA use: SET SCHEMA HOSPITAL_OPD_DB;

-- =====================================================
-- 2. CREATE TABLES
-- =====================================================

CREATE TABLE IF NOT EXISTS HOSPITAL_OPD_DB.PATIENTS (
    PATIENT_ID       INT PRIMARY KEY,
    PATIENT_NAME     VARCHAR(100) NOT NULL,
    GENDER           VARCHAR(10) NOT NULL,
    AGE              INT NOT NULL,
    CITY             VARCHAR(50) NOT NULL,
    MOBILE_NUMBER    VARCHAR(15) UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS HOSPITAL_OPD_DB.DEPARTMENTS (
    DEPARTMENT_ID    INT PRIMARY KEY,
    DEPARTMENT_NAME  VARCHAR(100) NOT NULL UNIQUE,
    FLOOR_NUMBER     INT NOT NULL
);

CREATE TABLE IF NOT EXISTS HOSPITAL_OPD_DB.DOCTORS (
    DOCTOR_ID          INT PRIMARY KEY,
    DOCTOR_NAME        VARCHAR(100) NOT NULL,
    DEPARTMENT_ID      INT NOT NULL,
    SPECIALIZATION     VARCHAR(100) NOT NULL,
    CONSULTATION_FEE   DECIMAL(10,2) NOT NULL,
    CONSTRAINT FK_DOCTOR_DEPARTMENT
        FOREIGN KEY (DEPARTMENT_ID)
        REFERENCES HOSPITAL_OPD_DB.DEPARTMENTS(DEPARTMENT_ID)
);

CREATE TABLE IF NOT EXISTS HOSPITAL_OPD_DB.APPOINTMENTS (
    APPOINTMENT_ID      INT PRIMARY KEY,
    PATIENT_ID          INT NOT NULL,
    DOCTOR_ID           INT NOT NULL,
    APPOINTMENT_DATE    DATE NOT NULL,
    APPOINTMENT_TIME    TIME NOT NULL,
    APPOINTMENT_STATUS  VARCHAR(20) NOT NULL,
    CONSTRAINT CHK_APPOINTMENT_STATUS
        CHECK (APPOINTMENT_STATUS IN ('Completed', 'Pending', 'Cancelled')),
    CONSTRAINT FK_APPOINTMENT_PATIENT
        FOREIGN KEY (PATIENT_ID)
        REFERENCES HOSPITAL_OPD_DB.PATIENTS(PATIENT_ID),
    CONSTRAINT FK_APPOINTMENT_DOCTOR
        FOREIGN KEY (DOCTOR_ID)
        REFERENCES HOSPITAL_OPD_DB.DOCTORS(DOCTOR_ID)
);

CREATE TABLE IF NOT EXISTS HOSPITAL_OPD_DB.BILLING (
    BILL_ID           INT PRIMARY KEY,
    APPOINTMENT_ID    INT NOT NULL UNIQUE,
    BILL_AMOUNT       DECIMAL(10,2) NOT NULL,
    PAYMENT_MODE      VARCHAR(20) NOT NULL,
    PAYMENT_STATUS    VARCHAR(20) NOT NULL,
    CONSTRAINT CHK_PAYMENT_MODE
        CHECK (PAYMENT_MODE IN ('Cash', 'UPI', 'Card', 'Insurance')),
    CONSTRAINT CHK_PAYMENT_STATUS
        CHECK (PAYMENT_STATUS IN ('Paid', 'Unpaid', 'Refunded')),
    CONSTRAINT FK_BILLING_APPOINTMENT
        FOREIGN KEY (APPOINTMENT_ID)
        REFERENCES HOSPITAL_OPD_DB.APPOINTMENTS(APPOINTMENT_ID)
);

-- =====================================================
-- 3. INSERT SAMPLE DATA
-- =====================================================

INSERT INTO HOSPITAL_OPD_DB.PATIENTS (PATIENT_ID, PATIENT_NAME, GENDER, AGE, CITY, MOBILE_NUMBER) VALUES
(1, 'Amit Sharma', 'Male', 34, 'Delhi', '9876500001'),
(2, 'Neha Verma', 'Female', 29, 'Noida', '9876500002'),
(3, 'Rohan Gupta', 'Male', 41, 'Ghaziabad', '9876500003'),
(4, 'Priya Singh', 'Female', 36, 'Delhi', '9876500004'),
(5, 'Karan Mehta', 'Male', 52, 'Faridabad', '9876500005'),
(6, 'Sneha Joshi', 'Female', 24, 'Ghaziabad', '9876500006');

INSERT INTO HOSPITAL_OPD_DB.DEPARTMENTS (DEPARTMENT_ID, DEPARTMENT_NAME, FLOOR_NUMBER) VALUES
(101, 'Cardiology', 2),
(102, 'Orthopedics', 3),
(103, 'General Medicine', 1),
(104, 'Pediatrics', 4);

INSERT INTO HOSPITAL_OPD_DB.DOCTORS (DOCTOR_ID, DOCTOR_NAME, DEPARTMENT_ID, SPECIALIZATION, CONSULTATION_FEE) VALUES
(1001, 'Dr. Arvind Rao', 101, 'Cardiologist', 800.00),
(1002, 'Dr. Meera Nair', 102, 'Orthopedic Specialist', 700.00),
(1003, 'Dr. Sandeep Khanna', 103, 'Physician', 500.00),
(1004, 'Dr. Pooja Bansal', 104, 'Pediatrician', 600.00),
(1005, 'Dr. Vivek Malhotra', 101, 'Heart Specialist', 900.00);

INSERT INTO HOSPITAL_OPD_DB.APPOINTMENTS (APPOINTMENT_ID, PATIENT_ID, DOCTOR_ID, APPOINTMENT_DATE, APPOINTMENT_TIME, APPOINTMENT_STATUS) VALUES
(5001, 1, 1001, '2026-06-25', '10:00:00', 'Completed'),
(5002, 2, 1002, '2026-06-25', '11:00:00', 'Completed'),
(5003, 3, 1003, '2026-06-26', '09:30:00', 'Pending'),
(5004, 4, 1004, '2026-06-26', '12:00:00', 'Cancelled'),
(5005, 5, 1005, '2026-06-27', '14:00:00', 'Completed'),
(5006, 6, 1003, '2026-06-27', '15:30:00', 'Completed'),
(5007, 1, 1002, '2026-06-28', '10:30:00', 'Pending'),
(5008, 2, 1001, '2026-06-28', '16:00:00', 'Completed');

INSERT INTO HOSPITAL_OPD_DB.BILLING (BILL_ID, APPOINTMENT_ID, BILL_AMOUNT, PAYMENT_MODE, PAYMENT_STATUS) VALUES
(9001, 5001, 800.00, 'UPI', 'Paid'),
(9002, 5002, 700.00, 'Card', 'Paid'),
(9003, 5003, 500.00, 'Cash', 'Unpaid'),
(9004, 5004, 600.00, 'Cash', 'Refunded'),
(9005, 5005, 900.00, 'Insurance', 'Paid'),
(9006, 5006, 500.00, 'UPI', 'Paid'),
(9007, 5007, 700.00, 'Card', 'Unpaid'),
(9008, 5008, 800.00, 'Cash', 'Paid');

-- =====================================================
-- 4. MAIN ANALYTICS VIEW
-- =====================================================

CREATE OR REPLACE VIEW HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS AS
SELECT
    A.APPOINTMENT_ID,
    A.APPOINTMENT_DATE,
    A.APPOINTMENT_TIME,
    P.PATIENT_NAME,
    P.CITY AS PATIENT_CITY,
    D.DOCTOR_NAME,
    DP.DEPARTMENT_NAME,
    D.SPECIALIZATION,
    D.CONSULTATION_FEE,
    B.BILL_AMOUNT,
    B.PAYMENT_MODE,
    B.PAYMENT_STATUS,
    A.APPOINTMENT_STATUS
FROM HOSPITAL_OPD_DB.APPOINTMENTS A
INNER JOIN HOSPITAL_OPD_DB.PATIENTS P
    ON A.PATIENT_ID = P.PATIENT_ID
INNER JOIN HOSPITAL_OPD_DB.DOCTORS D
    ON A.DOCTOR_ID = D.DOCTOR_ID
INNER JOIN HOSPITAL_OPD_DB.DEPARTMENTS DP
    ON D.DEPARTMENT_ID = DP.DEPARTMENT_ID
LEFT JOIN HOSPITAL_OPD_DB.BILLING B
    ON A.APPOINTMENT_ID = B.APPOINTMENT_ID;

-- =====================================================
-- 5. FINAL ASSIGNMENT VIEW
-- =====================================================

CREATE OR REPLACE VIEW HOSPITAL_OPD_DB.V_DEPARTMENT_DAILY_REVENUE AS
SELECT
    A.APPOINTMENT_DATE,
    DP.DEPARTMENT_NAME,
    COUNT(A.APPOINTMENT_ID) AS TOTAL_APPOINTMENTS,
    COUNT(B.BILL_ID) AS TOTAL_PAID_BILLS,
    SUM(B.BILL_AMOUNT) AS TOTAL_REVENUE
FROM HOSPITAL_OPD_DB.APPOINTMENTS A
INNER JOIN HOSPITAL_OPD_DB.DOCTORS D
    ON A.DOCTOR_ID = D.DOCTOR_ID
INNER JOIN HOSPITAL_OPD_DB.DEPARTMENTS DP
    ON D.DEPARTMENT_ID = DP.DEPARTMENT_ID
INNER JOIN HOSPITAL_OPD_DB.BILLING B
    ON A.APPOINTMENT_ID = B.APPOINTMENT_ID
WHERE A.APPOINTMENT_STATUS = 'Completed'
  AND B.PAYMENT_STATUS = 'Paid'
GROUP BY A.APPOINTMENT_DATE, DP.DEPARTMENT_NAME;

-- =====================================================
-- 6. BUSINESS QUESTIONS
-- =====================================================

-- Question 1: All completed OPD appointments
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Completed';

-- Question 2: Total OPD revenue generated by the hospital
SELECT
    SUM(BILL_AMOUNT) AS TOTAL_OPD_REVENUE
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid';

-- Question 3: Department-wise revenue
SELECT
    DEPARTMENT_NAME,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DEPARTMENT_NAME
ORDER BY TOTAL_REVENUE DESC;

-- Question 4: Doctor-wise consultation revenue
SELECT
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    COUNT(APPOINTMENT_ID) AS TOTAL_APPOINTMENTS,
    SUM(BILL_AMOUNT) AS TOTAL_REVENUE
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Paid'
GROUP BY DOCTOR_NAME, DEPARTMENT_NAME
ORDER BY TOTAL_REVENUE DESC;

-- Question 5: All pending appointments
SELECT
    APPOINTMENT_ID,
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    APPOINTMENT_DATE,
    APPOINTMENT_TIME
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE APPOINTMENT_STATUS = 'Pending';

-- Question 6: Unpaid bills
SELECT
    PATIENT_NAME,
    DOCTOR_NAME,
    DEPARTMENT_NAME,
    BILL_AMOUNT,
    PAYMENT_STATUS
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
WHERE PAYMENT_STATUS = 'Unpaid';

-- Question 7: City-wise patient visits
SELECT
    PATIENT_CITY,
    COUNT(APPOINTMENT_ID) AS TOTAL_NUMBER_OF_APPOINTMENTS
FROM HOSPITAL_OPD_DB.V_OPD_APPOINTMENT_ANALYTICS
GROUP BY PATIENT_CITY
ORDER BY TOTAL_NUMBER_OF_APPOINTMENTS DESC, PATIENT_CITY;
'''

print(sql_script[:2000])

## SQL script

The next cell stores the complete SQL solution in the variable `sql_script`.
You can copy it directly into your SQL editor, SAP HANA database explorer, or any compatible RDBMS tool.

In [ ]:
# Save the SQL script to a .sql file as well.
from pathlib import Path
Path('output/hospital_opd_solution.sql').write_text(sql_script, encoding='utf-8')
len(sql_script)

## Optional notes for SAP HANA users

- Replace `CREATE SCHEMA IF NOT EXISTS` with plain `CREATE SCHEMA HOSPITAL_OPD_DB;` if needed.
- Use `SET SCHEMA HOSPITAL_OPD_DB;` before table creation if your environment expects current-schema context.
- If `CREATE OR REPLACE VIEW` is not supported in your setup, first `DROP VIEW` and then `CREATE VIEW`.

## Short explanation

- `PATIENTS` stores patient master data.
- `DEPARTMENTS` stores department master data.
- `DOCTORS` links each doctor to a department and fee.
- `APPOINTMENTS` stores transactional visit data.
- `BILLING` stores one billing record per appointment.
- `V_OPD_APPOINTMENT_ANALYTICS` gives the consolidated reporting layer.
- `V_DEPARTMENT_DAILY_REVENUE` gives daily paid revenue for completed appointments.